<a href="https://colab.research.google.com/github/sunmgr/ml-projects/blob/main/KbinsDiscretizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [57]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer



In [58]:
df = pd.read_csv("train.csv",usecols= ['Age',"Fare","Survived"])

In [59]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [60]:
df.dropna(inplace=True)

In [61]:
df.shape

(714, 3)

In [62]:
X=df.iloc[:,1:]
y = df.iloc[:,0]

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [64]:
clf  = DecisionTreeClassifier()

In [65]:
clf.fit(X_train,y_train)

DecisionTreeClassifier()

In [66]:
y_pred = clf.predict(X_test)

In [67]:
accuracy_score(y_test,y_pred)

0.635593220338983

In [68]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6303403755868544)

In [69]:
kbin_age = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')
kbin_fare = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')

In [70]:
trf = ColumnTransformer([
  ("first",kbin_age,[0]),
  ("second",kbin_fare,[1])
]
)

In [71]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf  = trf.transform(X_test)

In [72]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.67,  7.8 , 17.  , 19.  , 22.  , 24.  , 25.  , 28.  , 30.  ,
              32.1 , 35.  , 39.  , 42.  , 48.  , 54.  , 80.  ])             ],
      dtype=object)

In [73]:
output = pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

In [74]:
output

,age,age_trf,fare,fare_trf
271,25.0,6.0,0.0000,0.0
853,16.0,1.0,39.4000,11.0
696,44.0,12.0,8.0500,3.0
518,36.0,10.0,26.0000,9.0
609,40.0,11.0,153.4625,14.0
...,...,...,...,...
92,46.0,12.0,61.1750,12.0
134,25.0,6.0,13.0000,6.0
337,41.0,11.0,134.5000,14.0
548,33.0,9.0,20.5250,8.0


In [75]:
output['age_labels'] = pd.cut(x=X_train['Age'],
                                    bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=X_train['Fare'],
                                    bins=trf.named_transformers_['second'].bin_edges_[0].tolist())


In [76]:
output

,age,age_trf,fare,fare_trf,age_labels,fare_labels
271,25.0,6.0,0.0000,0.0,"(24.0, 25.0]",NaN
853,16.0,1.0,39.4000,11.0,"(7.8, 17.0]","(32.864, 52.0]"
696,44.0,12.0,8.0500,3.0,"(42.0, 48.0]","(7.896, 8.072]"
518,36.0,10.0,26.0000,9.0,"(35.0, 39.0]","(19.785, 26.0]"
609,40.0,11.0,153.4625,14.0,"(39.0, 42.0]","(110.883, 263.0]"
...,...,...,...,...,...,...
92,46.0,12.0,61.1750,12.0,"(42.0, 48.0]","(52.0, 76.729]"
134,25.0,6.0,13.0000,6.0,"(24.0, 25.0]","(10.5, 13.0]"
337,41.0,11.0,134.5000,14.0,"(39.0, 42.0]","(110.883, 263.0]"
548,33.0,9.0,20.5250,8.0,"(32.1, 35.0]","(19.785, 26.0]"


In [78]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(x_test_trf)

In [79]:
accuracy_score(y_test,y_pred2)

0.6228813559322034